# Part A - Baseline Model

In [ ]:
# %% [markdown]
# # 03 - Baseline Model Training
# **Module:** Deep Learning (MOD006565)
# **Description:** Define and train a simplified Inception-based baseline model inspired
# by the DeepFood paper (Liu et al., 2016). Evaluate on validation and test sets.

# %% [markdown]
# ## Imports

# %%
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image

# %% [markdown]
# ## Configuration

# %%
SPLIT_DIR      = '/workspace/food101-dl-project/splits'
OUTPUT_DIR     = '/workspace/food101-dl-project/outputs'
CHECKPOINT_DIR = '/workspace/food101-dl-project/checkpoints'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE  = 64
IMAGE_SIZE  = 128
NUM_EPOCHS  = 30
LR          = 0.01
MOMENTUM    = 0.9
DROPOUT     = 0.7
WEIGHT_DECAY = 1e-4

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device         : {device}")
if torch.cuda.is_available():
    print(f"GPU            : {torch.cuda.get_device_name(0)}")

# %% [markdown]
# ## Load Class Mapping and Splits

# %%
with open(f'{SPLIT_DIR}/class_mapping.json', 'r') as f:
    mapping = json.load(f)

SELECTED_CLASSES = mapping['selected_classes']
NUM_CLASSES      = len(SELECTED_CLASSES)

train_images = np.load(f'{SPLIT_DIR}/train_images.npy')
train_labels = np.load(f'{SPLIT_DIR}/train_labels.npy')
val_images   = np.load(f'{SPLIT_DIR}/val_images.npy')
val_labels   = np.load(f'{SPLIT_DIR}/val_labels.npy')
test_images  = np.load(f'{SPLIT_DIR}/test_images.npy')
test_labels  = np.load(f'{SPLIT_DIR}/test_labels.npy')

print(f"Classes        : {NUM_CLASSES}")
print(f"Train          : {len(train_images)}")
print(f"Validation     : {len(val_images)}")
print(f"Test           : {len(test_images)}")

# %% [markdown]
# ## Dataset and DataLoaders

# %%
class Food101Dataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images    = images
        self.labels    = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image = Image.fromarray(self.images[idx].astype(np.uint8))
        label = int(self.labels[idx])
        if self.transform:
            image = self.transform(image)
        return image, label


train_transform = transforms.Compose([
    transforms.Resize((140, 140)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(Food101Dataset(train_images, train_labels, train_transform),
                          batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(Food101Dataset(val_images,   val_labels,   val_test_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(Food101Dataset(test_images,  test_labels,  val_test_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train batches      : {len(train_loader)}")
print(f"Validation batches : {len(val_loader)}")
print(f"Test batches       : {len(test_loader)}")

# %% [markdown]
# ## Model Architecture
#
# The baseline model is directly inspired by the DeepFood paper (Liu et al., 2016),
# which proposes a GoogLeNet-based architecture using Inception modules for food
# image classification on the Food-101 dataset.
#
# Key design decisions following the paper:
# - Inception modules with parallel 1x1, 3x3, 5x5 convolutions and max pooling
# - 1x1 convolutions for dimensionality reduction before 3x3 and 5x5 branches
# - Global average pooling instead of fully connected spatial layers
# - 1024-dimensional feature layer as in the paper
# - 70% dropout rate as specified in the paper
# - ReLU activations throughout
# - Softmax loss (implemented as CrossEntropyLoss in PyTorch)
#
# Adaptations made for computational constraints:
# - Input size reduced from 224x224 to 128x128
# - Two Inception modules instead of nine (as in full GoogLeNet)
# - 50 output classes instead of 101

# %%
class InceptionModule(nn.Module):
    """
    Inception module as described in the DeepFood paper.
    Parallel branches of 1x1, 3x3, 5x5 convolutions and max pooling
    are concatenated to capture multi-scale spatial features.
    """
    def __init__(self, in_channels, out_1x1, out_3x3, out_5x5, out_pool):
        super(InceptionModule, self).__init__()

        self.branch_1x1 = nn.Sequential(
            nn.Conv2d(in_channels, out_1x1, kernel_size=1),
            nn.BatchNorm2d(out_1x1),
            nn.ReLU(inplace=True)
        )

        self.branch_3x3 = nn.Sequential(
            nn.Conv2d(in_channels, out_1x1, kernel_size=1),
            nn.BatchNorm2d(out_1x1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_1x1, out_3x3, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_3x3),
            nn.ReLU(inplace=True)
        )

        self.branch_5x5 = nn.Sequential(
            nn.Conv2d(in_channels, out_1x1, kernel_size=1),
            nn.BatchNorm2d(out_1x1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_1x1, out_5x5, kernel_size=5, padding=2),
            nn.BatchNorm2d(out_5x5),
            nn.ReLU(inplace=True)
        )

        self.branch_pool = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, out_pool, kernel_size=1),
            nn.BatchNorm2d(out_pool),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        b1 = self.branch_1x1(x)
        b2 = self.branch_3x3(x)
        b3 = self.branch_5x5(x)
        b4 = self.branch_pool(x)
        return torch.cat([b1, b2, b3, b4], dim=1)


class DeepFoodBaseline(nn.Module):
    """
    Simplified GoogLeNet-inspired model following the DeepFood paper.

    Architecture:
        Stem  : Conv(7x7) -> MaxPool -> Conv(1x1) -> Conv(3x3) -> MaxPool
        Block1: InceptionModule -> MaxPool
        Block2: InceptionModule -> MaxPool
        Head  : GlobalAvgPool -> Dropout(0.7) -> FC(1024) -> Dropout(0.5) -> FC(num_classes)
    """
    def __init__(self, num_classes=50, dropout=0.7):
        super(DeepFoodBaseline, self).__init__()

        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            nn.Conv2d(64, 64, kernel_size=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )

        self.inception1 = InceptionModule(192, 64, 128, 32, 32)
        self.maxpool1   = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.inception2 = InceptionModule(256, 128, 192, 96, 64)
        self.maxpool2   = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.global_avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(480, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.inception1(x)
        x = self.maxpool1(x)
        x = self.inception2(x)
        x = self.maxpool2(x)
        x = self.global_avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


model = DeepFoodBaseline(num_classes=NUM_CLASSES, dropout=DROPOUT).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model              : DeepFood Baseline (Inception-inspired)")
print(f"Total parameters   : {total_params:,}")
print(f"Trainable params   : {trainable_params:,}")
print(f"Input size         : {IMAGE_SIZE}x{IMAGE_SIZE}x3")
print(f"Output classes     : {NUM_CLASSES}")
print(f"Dropout rate       : {DROPOUT}")

# %% [markdown]
# ## Loss Function, Optimiser and Scheduler
#
# Following the DeepFood paper:
# - CrossEntropyLoss is equivalent to the Softmax loss used in the paper
# - SGD with momentum=0.9 and base learning rate=0.01 as specified
# - StepLR scheduler decays the learning rate to prevent overshooting

# %%
criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=LR,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

print(f"Loss function      : CrossEntropyLoss")
print(f"Optimiser          : SGD (lr={LR}, momentum={MOMENTUM}, weight_decay={WEIGHT_DECAY})")
print(f"Scheduler          : StepLR (step_size=10, gamma=0.1)")

# %% [markdown]
# ## Training and Evaluation Functions

# %%
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return total_loss / total, 100.0 * correct / total


def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total   += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return total_loss / total, 100.0 * correct / total

# %% [markdown]
# ## Training Loop

# %%
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss':   [], 'val_acc':   []
}

best_val_acc = 0.0

print(f"Training for {NUM_EPOCHS} epochs on {device}")
print(f"{'Epoch':<8} {'Train Loss':<14} {'Train Acc':<14} {'Val Loss':<14} {'Val Acc':<14} {'Time (s)'}")
print("-" * 78)

for epoch in range(NUM_EPOCHS):
    start = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_acc   = eval_epoch(model, val_loader,   criterion, device)

    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    elapsed = time.time() - start

    print(f"{epoch+1:<8} {train_loss:<14.4f} {train_acc:<14.2f} {val_loss:<14.4f} {val_acc:<14.2f} {elapsed:.1f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), f'{CHECKPOINT_DIR}/baseline_best.pth')
        print(f"         Best model saved at epoch {epoch+1} with val accuracy {best_val_acc:.2f}%")

print("-" * 78)
print(f"Training complete. Best validation accuracy : {best_val_acc:.2f}%")

# %% [markdown]
# ## Training Curves

# %%
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'], label='Train Loss',      color='steelblue')
ax1.plot(history['val_loss'],   label='Validation Loss', color='darkorange')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history['train_acc'], label='Train Accuracy',      color='steelblue')
ax2.plot(history['val_acc'],   label='Validation Accuracy', color='darkorange')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('DeepFood Baseline — Training Curves', fontsize=13)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/baseline_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Training curves saved to {OUTPUT_DIR}/baseline_training_curves.png")

# %% [markdown]
# ## Final Evaluation on Test Set

# %%
model.load_state_dict(torch.load(f'{CHECKPOINT_DIR}/baseline_best.pth'))
test_loss, test_acc = eval_epoch(model, test_loader, criterion, device)

print("Test Set Evaluation — DeepFood Baseline")
print("-" * 40)
print(f"Test Loss          : {test_loss:.4f}")
print(f"Test Accuracy      : {test_acc:.2f}%")
print(f"Best Val Accuracy  : {best_val_acc:.2f}%")
print("-" * 40)

results = {
    'model':            'DeepFood Baseline (Inception-inspired)',
    'num_classes':      NUM_CLASSES,
    'input_size':       f'{IMAGE_SIZE}x{IMAGE_SIZE}',
    'epochs':           NUM_EPOCHS,
    'optimizer':        f'SGD lr={LR} momentum={MOMENTUM}',
    'dropout':          DROPOUT,
    'best_val_acc':     round(best_val_acc, 4),
    'test_accuracy':    round(test_acc, 4),
    'test_loss':        round(test_loss, 4),
    'train_history':    history
}

with open(f'{OUTPUT_DIR}/baseline_results.json', 'w') as f:
    json.dump(results, f, indent=4)

print(f"Results saved to {OUTPUT_DIR}/baseline_results.json")